In [1]:
# Step 1 — Copy/extract the assignment into writable Kaggle working storage.

import glob
import os
import shutil
import zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/PA2")
WORK_DIR.parent.mkdir(parents=True, exist_ok=True)

if (WORK_DIR / "pyproject.toml").exists():
    print("PA2 project already exists:", WORK_DIR)
else:
    # Option A: the uploaded Kaggle dataset already contains the extracted project.
    project_files = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    if project_files:
        project_root = Path(project_files[0]).parent
        print("Found extracted project:", project_root)
        shutil.copytree(project_root, WORK_DIR, dirs_exist_ok=True)
    else:
        # Option B: the Kaggle dataset contains PA2.zip.
        zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if not zip_files:
            raise FileNotFoundError(
                "Could not find PA2.zip or a pyproject.toml under /kaggle/input. "
                "Use Add Input to attach the PA2 dataset."
            )
        zip_path = Path(zip_files[0])
        print("Extracting:", zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall("/kaggle/working")
        if not (WORK_DIR / "pyproject.toml").exists():
            # Handle archives whose top-level folder name is not exactly PA2.
            extracted_projects = glob.glob("/kaggle/working/**/pyproject.toml", recursive=True)
            extracted_projects = [
                p for p in extracted_projects
                if not p.startswith("/kaggle/working/PA2/")
            ]
            if extracted_projects:
                shutil.copytree(
                    str(Path(extracted_projects[0]).parent),
                    WORK_DIR,
                    dirs_exist_ok=True
                )
            else:
                raise FileNotFoundError(
                    "Extraction finished, but PA2/pyproject.toml was not found. "
                    "Inspect the ZIP's folder structure."
                )

os.chdir(WORK_DIR)

for folder in (
    "outputs/gen",
    "outputs/eval",
    "outputs/sft",
    "outputs/dpo_hh",
    "outputs/dpo_uf",
    "eval_sets",
):
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project directory:", os.getcwd())
print("Top-level files:", sorted(os.listdir("."))[:30])


Found extracted project: /kaggle/input/datasets/areeshayaseen/cs5326-pa2/PA2
Project directory: /kaggle/working/PA2
Top-level files: ['.gitignore', 'README.md', 'data', 'docs', 'eval_sets', 'outputs', 'pa2', 'pyproject.toml', 'requirements.txt', 'scripts', 'submission_adapter.py', 'tests']


**Install Project Dependencies**

In [2]:
%cd /kaggle/working/PA2

!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
!python -m pip install -q -e . --no-deps

!nvidia-smi -L

import torch
import transformers
import datasets

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    print("WARNING: CUDA is unavailable. Check Kaggle's Accelerator setting.")


/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for small-model-alignment-starter (py

**Run the public adapter smoke tests**

In [3]:
%cd /kaggle/working/PA2
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py

/kaggle/working
.......                                                                  [100%]
7 passed in 2.26s


**Create the fixed AlpacaEval subset and run a 3-prompt sanity check**

In [4]:
%cd /kaggle/working/PA2
!python scripts/gen_alpaca_subset.py
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --out_path outputs/gen/base_sanity.json --generator qwen2.5-0.5b-base --limit 3

import json
sanity_path = "outputs/gen/base_sanity.json"
with open(sanity_path, "r", encoding="utf-8") as f:
    sanity_rows = json.load(f)

print("Sanity outputs:", len(sanity_rows))
for row in sanity_rows:
    print("\nINSTRUCTION:", row.get("instruction", "")[:200])
    print("OUTPUT:", row.get("output", "")[:500])


/kaggle/working
alpaca_eval.json: 621kB [00:00, 31.5MB/s]
saved 300 prompts -> eval_sets/alpaca_eval_300_seed42.json
tokenizer_config.json: 7.23kB [00:00, 21.2MB/s]
vocab.json: 2.78MB [00:00, 75.1MB/s]
merges.txt: 1.67MB [00:00, 124MB/s]
tokenizer.json: 7.03MB [00:00, 183MB/s]
config.json: 100%|█████████████████████████████| 681/681 [00:00<00:00, 2.65MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100%|██████████████████████| 988M/988M [00:04<00:00, 232MB/s]
generation_config.json: 100%|███████████████████| 138/138 [00:00<00:00, 552kB/s]
3/3 done
Sanity outputs: 3

INSTRUCTION: How do I dice without slicing my finger
OUTPUT: To dice without slicing your finger, you can follow these steps:

1. **Prepare the ingredients**: Ensure that you have all the necessary ingredients for the dice, such as flour, salt, pepper, and any other spices or seasonings you may need.

2. **Mix the ingredients**: In a large mixing bowl, combine the flour, salt, and pepper. Mix well 

**Full Base-Model Generation**

In [5]:
%cd /kaggle/working/PA2
G = "--max_new_tokens 384 --repetition_penalty 1.1 --no_repeat_ngram_size 0"
!python scripts/generate.py --model_path Qwen/Qwen2.5-0.5B --eval_path eval_sets/alpaca_eval_300_seed42.json --out_path outputs/gen/base_alpaca.json --generator qwen2.5-0.5b-base --max_new_tokens 384 --repetition_penalty 1.1 --no_repeat_ngram_size 0


/kaggle/working
`torch_dtype` is deprecated! Use `dtype` instead!
16/300 done
32/300 done
48/300 done
64/300 done
80/300 done
96/300 done
112/300 done
128/300 done
144/300 done
160/300 done
176/300 done
192/300 done
208/300 done
224/300 done
240/300 done
256/300 done
272/300 done
288/300 done
300/300 done


**Verify and Package Outputs**

In [6]:
%cd /kaggle/working/PA2

import os
from pathlib import Path

for path in [
    "outputs/gen/base_sanity.json",
    "outputs/gen/base_alpaca.json",
    "eval_sets/alpaca_eval_300_seed42.json",
]:
    print(f"{path}: {'FOUND' if Path(path).exists() else 'MISSING'}")

# Package the outputs needed for inspection and handoff.
%cd /kaggle/working
!zip -rq PA2_notebook1_outputs.zip PA2/outputs PA2/eval_sets
print("Created /kaggle/working/PA2_notebook1_outputs.zip")


/kaggle/working
outputs/gen/base_sanity.json: FOUND
outputs/gen/base_alpaca.json: FOUND
eval_sets/alpaca_eval_300_seed42.json: FOUND
/kaggle/working
Created /kaggle/working/PA2_notebook1_outputs.zip
